# HealthBot: AI-Powered Patient Education System

**MediTech Solutions — Healthcare AI Capstone Prototype**

This notebook implements a LangGraph workflow for **HealthBot**, a chatbot that:

1. Asks the patient which health topic they want to learn about.
2. Searches the web via **Tavily** (through the LangChain community tool) for up-to-date, reputable medical information.
3. Summarizes the search results into a **patient-friendly** explanation, using only the retrieved content.
4. Lets the patient read the summary, then gives a one-question **comprehension check**.
5. Grades the patient's answer against the summary and explains the grade with citations.
6. Lets the patient loop into a new topic (with a clean, reset state) or exit.

Run each cell top-to-bottom. The final cell starts an interactive session — patient input is collected
with Python's `input()` (which Jupyter renders as a modal input box), and output is shown with `print()`.


## 1. Setup: environment variables

Create a `config.env` file (see `config.env.example`) in this project's directory with your own keys:

```env
OPENAI_API_KEY="sk-..."
TAVILY_API_KEY="tvly-..."
```

Never commit `config.env` — it's already excluded via `.gitignore`.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None, (
    "OPENAI_API_KEY is not set. Create config.env from config.env.example and add your key."
)
assert os.getenv("TAVILY_API_KEY") is not None, (
    "TAVILY_API_KEY is not set. Create config.env from config.env.example and add your key."
)

print("Environment variables loaded successfully.")


## 2. Imports and model configuration

We use:
- `ChatOpenAI` as the LLM.
- `TavilySearchResults` (the LangChain **community** Tavily tool) restricted to a set of reputable
  medical domains, so search results stay clinically trustworthy.
- LangGraph's `StateGraph` to wire the workflow together.


In [ ]:
from typing import Optional, TypedDict

from langchain_core.messages import HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

MODEL_NAME = os.getenv("HEALTHBOT_MODEL", "gpt-4o-mini")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

# Reputable medical sources only -- keeps the bot from summarizing random, low-quality web pages.
REPUTABLE_MEDICAL_DOMAINS = [
    "mayoclinic.org",
    "nih.gov",
    "cdc.gov",
    "medlineplus.gov",
    "who.int",
    "webmd.com",
    "clevelandclinic.org",
]

tavily_tool = TavilySearchResults(
    max_results=5,
    include_domains=REPUTABLE_MEDICAL_DOMAINS,
)

print(f"LLM configured: {MODEL_NAME}")
print(f"Tavily tool configured: {tavily_tool.name}, restricted to {len(REPUTABLE_MEDICAL_DOMAINS)} reputable domains")


## 3. Workflow state

A single `HealthBotState` `TypedDict` is threaded through every node. Each node reads what it needs from
state and returns only the fields it updates -- LangGraph merges those updates in.

`exit_session` is the only field that survives a topic-to-topic reset; everything else about the previous
topic (search results, summary, quiz, answer, grade) is cleared out when a new topic starts, so a
patient's new question is never influenced by -- or mixed up with -- a previous patient's medical topic.


In [ ]:
class HealthBotState(TypedDict, total=False):
    topic: str
    search_results: str
    summary: str
    quiz_question: str
    user_answer: str
    grade: str
    justification: str
    exit_session: bool


class GradeResult(BaseModel):
    """Structured grade for a patient's quiz answer."""

    grade: str = Field(description="A single letter grade: A, B, C, D, or F.")
    justification: str = Field(
        description=(
            "Explanation for the grade, quoting or closely paraphrasing the specific "
            "sentence(s) from the summary that support the correct answer."
        )
    )


## 4. Node functions

Each node has a single responsibility: collect input, call a tool, call the LLM, or display output.


### 4.1 Collect the health topic (and reset state for a fresh session)

In [ ]:
def collect_topic(state: HealthBotState) -> HealthBotState:
    topic = input("What health topic or medical condition would you like to learn about? ").strip()
    while not topic:
        topic = input("Please enter a health topic or medical condition: ").strip()

    # Reset every field from any previous topic so nothing leaks across sessions.
    return {
        "topic": topic,
        "search_results": "",
        "summary": "",
        "quiz_question": "",
        "user_answer": "",
        "grade": "",
        "justification": "",
        "exit_session": False,
    }


### 4.2 Search Tavily for reputable medical information (model calls the Tavily tool)

In [ ]:
def format_tavily_results(raw_results) -> str:
    if isinstance(raw_results, str):
        return raw_results

    formatted = []
    for i, result in enumerate(raw_results, start=1):
        title = result.get("title", "Untitled")
        url = result.get("url", "")
        content = result.get("content", "")
        formatted.append(f"Source {i}: {title} ({url})\n{content}")
    return "\n\n".join(formatted)


def search_health_info(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]

    llm_with_tools = llm.bind_tools([tavily_tool])
    instruction = (
        f"Search for current, reputable medical information about: {topic}. "
        "Focus on trustworthy medical sources (e.g. Mayo Clinic, CDC, NIH, MedlinePlus, WHO)."
    )
    ai_message = llm_with_tools.invoke([HumanMessage(content=instruction)])

    tool_outputs = []
    for tool_call in getattr(ai_message, "tool_calls", None) or []:
        if tool_call["name"] == tavily_tool.name:
            tool_outputs.append(tavily_tool.invoke(tool_call["args"]))

    # Fallback in the rare case the model doesn't emit a tool call.
    if not tool_outputs:
        tool_outputs.append(tavily_tool.invoke({"query": topic}))

    combined_results = "\n\n".join(format_tavily_results(r) for r in tool_outputs)
    return {"search_results": combined_results}


### 4.3 Summarize the search results in patient-friendly language

In [ ]:
SUMMARY_SYSTEM_PROMPT = (
    "You are a patient education assistant at a hospital. Using ONLY the medical search results "
    "provided below -- and no other knowledge or outside data sources -- write a clear, "
    "patient-friendly explanation of the topic. Avoid medical jargon where possible, and briefly "
    "explain any clinical terms you must use. Write exactly 3 to 4 paragraphs."
)


def summarize_results(state: HealthBotState) -> HealthBotState:
    prompt = (
        f"{SUMMARY_SYSTEM_PROMPT}\n\n"
        f"Topic: {state['topic']}\n\n"
        f"Search results:\n{state['search_results']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"summary": response.content}


### 4.4 Present the summary to the patient

In [ ]:
def display_summary(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 70)
    print(f"HERE'S WHAT WE FOUND ABOUT: {state['topic'].upper()}")
    print("=" * 70)
    print(state["summary"])
    print("=" * 70 + "\n")
    return {}


### 4.5 Wait for the patient to be ready for the comprehension check

In [ ]:
def await_ready_for_quiz(state: HealthBotState) -> HealthBotState:
    input("Take your time reading. Press Enter when you're ready for a quick comprehension check... ")
    return {}


### 4.6 Generate a one-question quiz from the summary only

In [ ]:
QUIZ_SYSTEM_PROMPT = (
    "You are writing a single comprehension-check question for a patient. Using ONLY the summary "
    "below -- not the original search results or any outside knowledge -- write exactly ONE quiz "
    "question that a patient could answer correctly after reading the summary. "
    "Respond with only the question text, nothing else."
)


def generate_quiz_question(state: HealthBotState) -> HealthBotState:
    prompt = f"{QUIZ_SYSTEM_PROMPT}\n\nSummary:\n{state['summary']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"quiz_question": response.content.strip()}


### 4.7 Present the quiz question

In [ ]:
def display_quiz_question(state: HealthBotState) -> HealthBotState:
    print("\n--- Comprehension Check ---")
    print(state["quiz_question"])
    print("---------------------------\n")
    return {}


### 4.8 Collect the patient's answer

In [ ]:
def collect_quiz_answer(state: HealthBotState) -> HealthBotState:
    answer = input("Your answer: ").strip()
    while not answer:
        answer = input("Please enter an answer: ").strip()
    return {"user_answer": answer}


### 4.9 Grade the answer using only the summary, with cited justification

In [ ]:
GRADE_SYSTEM_PROMPT = (
    "You are grading a patient's answer to a comprehension-check question. Use ONLY the summary "
    "below as your source of truth -- do not use outside medical knowledge. Grade the patient's "
    "answer with a single letter grade (A, B, C, D, or F) based on accuracy and completeness. "
    "In your justification, cite or closely paraphrase the specific sentence(s) from the summary "
    "that support the correct answer, and explain what the patient got right or missed."
)


def grade_quiz_answer(state: HealthBotState) -> HealthBotState:
    structured_llm = llm.with_structured_output(GradeResult)
    prompt = (
        f"{GRADE_SYSTEM_PROMPT}\n\n"
        f"Summary:\n{state['summary']}\n\n"
        f"Quiz question: {state['quiz_question']}\n\n"
        f"Patient's answer: {state['user_answer']}"
    )
    result: GradeResult = structured_llm.invoke([HumanMessage(content=prompt)])
    return {"grade": result.grade, "justification": result.justification}


### 4.10 Present the grade and feedback

In [ ]:
def display_grade(state: HealthBotState) -> HealthBotState:
    print("\n*** Your Grade ***")
    print(f"Grade: {state['grade']}")
    print(f"Feedback: {state['justification']}")
    print("*******************\n")
    return {}


### 4.11 Ask whether to continue with a new topic or exit

In [ ]:
def ask_continue_or_exit(state: HealthBotState) -> HealthBotState:
    while True:
        choice = input(
            "Would you like to learn about another health topic, or exit? (new/exit): "
        ).strip().lower()
        if choice in ("new", "n"):
            return {"exit_session": False}
        if choice in ("exit", "e", "quit", "q"):
            return {"exit_session": True}
        print("Please type 'new' or 'exit'.")


def route_continue_or_exit(state: HealthBotState) -> str:
    return END if state.get("exit_session") else "collect_topic"


## 5. Build the graph

Nodes are wired in the sequential order the patient experiences them. The only cycle is the loop back
from `ask_continue_or_exit` to `collect_topic`, which is also where per-topic state gets reset.


In [ ]:
graph = StateGraph(HealthBotState)

graph.add_node("collect_topic", collect_topic)
graph.add_node("search_health_info", search_health_info)
graph.add_node("summarize_results", summarize_results)
graph.add_node("display_summary", display_summary)
graph.add_node("await_ready_for_quiz", await_ready_for_quiz)
graph.add_node("generate_quiz_question", generate_quiz_question)
graph.add_node("display_quiz_question", display_quiz_question)
graph.add_node("collect_quiz_answer", collect_quiz_answer)
graph.add_node("grade_quiz_answer", grade_quiz_answer)
graph.add_node("display_grade", display_grade)
graph.add_node("ask_continue_or_exit", ask_continue_or_exit)

graph.add_edge(START, "collect_topic")
graph.add_edge("collect_topic", "search_health_info")
graph.add_edge("search_health_info", "summarize_results")
graph.add_edge("summarize_results", "display_summary")
graph.add_edge("display_summary", "await_ready_for_quiz")
graph.add_edge("await_ready_for_quiz", "generate_quiz_question")
graph.add_edge("generate_quiz_question", "display_quiz_question")
graph.add_edge("display_quiz_question", "collect_quiz_answer")
graph.add_edge("collect_quiz_answer", "grade_quiz_answer")
graph.add_edge("grade_quiz_answer", "display_grade")
graph.add_edge("display_grade", "ask_continue_or_exit")
graph.add_conditional_edges(
    "ask_continue_or_exit",
    route_continue_or_exit,
    {"collect_topic": "collect_topic", END: END},
)

app = graph.compile()
print("HealthBot graph compiled successfully.")


### Optional: visualize the graph


In [ ]:
try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualization unavailable ({e}); printing Mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())


## 6. Run HealthBot

This starts an interactive session. You'll be prompted for a topic, shown a summary, given a one-question
comprehension check, shown your grade, and asked if you'd like to continue with a new topic or exit.


In [ ]:
print("Welcome to HealthBot -- your on-demand patient education assistant.\n")

app.invoke({}, config={"recursion_limit": 200})

print("Thanks for using HealthBot. Take care!")
